In [1]:
import random
import time
from datetime import datetime, timedelta

import pandas as pd
from IPython.display import display, clear_output

In [2]:
MENU = [
    {
        "dish_name": "Pizza Margherita",
        "dish_category": "Pizza",
        "price": 29.90,
        "expected_preparation_seconds": 60
    },
    {
        "dish_name": "Pizza Pepperoni",
        "dish_category": "Pizza",
        "price": 34.90,
        "expected_preparation_seconds": 70
    },
    {
        "dish_name": "Pizza Capricciosa",
        "dish_category": "Pizza",
        "price": 36.90,
        "expected_preparation_seconds": 75
    },
    {
        "dish_name": "Burger klasyczny",
        "dish_category": "Burger",
        "price": 31.90,
        "expected_preparation_seconds": 40
    },
    {
        "dish_name": "Burger BBQ",
        "dish_category": "Burger",
        "price": 36.90,
        "expected_preparation_seconds": 45
    },
    {
        "dish_name": "Makaron Carbonara",
        "dish_category": "Makaron",
        "price": 32.90,
        "expected_preparation_seconds": 50
    },
    {
        "dish_name": "Makaron Bolognese",
        "dish_category": "Makaron",
        "price": 34.90,
        "expected_preparation_seconds": 55
    },
    {
        "dish_name": "Sałatka Cezar",
        "dish_category": "Sałatka",
        "price": 27.90,
        "expected_preparation_seconds": 25
    },
    {
        "dish_name": "Zupa dnia",
        "dish_category": "Zupa",
        "price": 18.90,
        "expected_preparation_seconds": 20
    },
    {
        "dish_name": "Frytki",
        "dish_category": "Dodatek",
        "price": 12.90,
        "expected_preparation_seconds": 15
    }
]

ORDER_CHANNELS = ["app", "website", "phone", "restaurant"]

In [3]:
def is_promotion_active():
    """
    Promocja aktywna w godzinach 18:00-20:59.
    Do testów możesz tymczasowo zmienić na: return True
    """
    current_hour = datetime.now().hour
    return 18 <= current_hour <= 20


def choose_dish():
    weights = [
        0.14,  # Pizza Margherita
        0.16,  # Pizza Pepperoni
        0.10,  # Pizza Capricciosa
        0.14,  # Burger klasyczny
        0.12,  # Burger BBQ
        0.10,  # Makaron Carbonara
        0.08,  # Makaron Bolognese
        0.07,  # Sałatka Cezar
        0.04,  # Zupa dnia
        0.05   # Frytki
    ]
    return random.choices(MENU, weights=weights, k=1)[0]


def generate_quantity():
    return random.choices(
        population=[1, 2, 3],
        weights=[0.65, 0.25, 0.10],
        k=1
    )[0]


def generate_actual_preparation_seconds(expected_seconds):
    """
    Rzeczywisty czas przygotowania.
    Czasami krótszy, czasami dłuższy od oczekiwanego.
    """
    delay = random.randint(-8, 20)
    actual_seconds = expected_seconds + delay
    return max(5, actual_seconds)


def generate_order(order_number):
    dish = choose_dish()
    quantity = generate_quantity()

    created_at = datetime.now()

    expected_seconds = dish["expected_preparation_seconds"]
    actual_seconds = generate_actual_preparation_seconds(expected_seconds)

    estimated_ready_at = created_at + timedelta(seconds=expected_seconds)
    actual_ready_at = created_at + timedelta(seconds=actual_seconds)

    is_delayed = actual_seconds > expected_seconds
    delay_seconds = max(0, actual_seconds - expected_seconds)

    order = {
        "order_id": f"ORD_{order_number:06d}",
        "timestamp": created_at,
        "created_at": created_at,

        "dish_name": dish["dish_name"],
        "dish_category": dish["dish_category"],

        "quantity": quantity,
        "unit_price": dish["price"],
        "order_value": round(dish["price"] * quantity, 2),
        "order_channel": random.choice(ORDER_CHANNELS),

        "expected_preparation_seconds": expected_seconds,
        "actual_preparation_seconds": actual_seconds,

        "estimated_ready_at": estimated_ready_at,
        "actual_ready_at": actual_ready_at,

        "is_delayed": is_delayed,
        "delay_seconds": delay_seconds,

        "status": "in_progress",
        "is_promotion": is_promotion_active()
    }

    return order

In [4]:
test_order = generate_order(1)
test_order

{'order_id': 'ORD_000001',
 'timestamp': datetime.datetime(2026, 6, 1, 19, 59, 40, 933702),
 'created_at': datetime.datetime(2026, 6, 1, 19, 59, 40, 933702),
 'dish_name': 'Pizza Margherita',
 'dish_category': 'Pizza',
 'quantity': 2,
 'unit_price': 29.9,
 'order_value': 59.8,
 'order_channel': 'website',
 'expected_preparation_seconds': 60,
 'actual_preparation_seconds': 54,
 'estimated_ready_at': datetime.datetime(2026, 6, 1, 20, 0, 40, 933702),
 'actual_ready_at': datetime.datetime(2026, 6, 1, 20, 0, 34, 933702),
 'is_delayed': False,
 'delay_seconds': 0,
 'status': 'in_progress',
 'is_promotion': True}

In [5]:
def show_dashboard(orders):
    clear_output(wait=True)

    if len(orders) == 0:
        print("Brak zamówień.")
        return

    df = pd.DataFrame(orders)

    now = pd.Timestamp.now()

    df["created_at"] = pd.to_datetime(df["created_at"])
    df["estimated_ready_at"] = pd.to_datetime(df["estimated_ready_at"])
    df["actual_ready_at"] = pd.to_datetime(df["actual_ready_at"])

    df["seconds_remaining"] = (
        df["actual_ready_at"] - now
    ).dt.total_seconds().round(0)

    df["current_status"] = df["seconds_remaining"].apply(
        lambda x: "in_progress" if x > 0 else "completed"
    )

    active_orders = df[df["current_status"] == "in_progress"].copy()
    completed_orders = df[df["current_status"] == "completed"].copy()

    total_orders = len(df)
    active_count = len(active_orders)
    completed_count = len(completed_orders)
    avg_preparation_time = df["actual_preparation_seconds"].mean()
    delayed_share = df["is_delayed"].mean() * 100
    total_revenue = df["order_value"].sum()

    if active_count < 5 and delayed_share < 30:
        status = "🟢 Zielony"
        status_description = "Sytuacja stabilna."
    elif active_count < 10 and delayed_share < 50:
        status = "🟡 Żółty"
        status_description = "Zwiększony ruch, warto monitorować kuchnię."
    else:
        status = "🔴 Czerwony"
        status_description = "Ryzyko przeciążenia kuchni."

    print("🍽️ SYMULACJA DASHBOARDU RESTAURACJI")
    print("=" * 70)
    print(f"Czas aktualny: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Status operacyjny: {status} — {status_description}")
    print("=" * 70)
    print(f"Wszystkie zamówienia: {total_orders}")
    print(f"Aktywne zamówienia: {active_count}")
    print(f"Zakończone zamówienia: {completed_count}")
    print(f"Średni czas przygotowania: {avg_preparation_time:.0f} s")
    print(f"Udział opóźnień: {delayed_share:.1f}%")
    print(f"Przychód: {total_revenue:.2f} zł")
    print("=" * 70)

    print("\n🔥 AKTYWNE ZAMÓWIENIA")
    if len(active_orders) > 0:
        active_view = active_orders[
            [
                "order_id",
                "dish_name",
                "dish_category",
                "quantity",
                "actual_preparation_seconds",
                "seconds_remaining",
                "is_delayed",
                "is_promotion"
            ]
        ].sort_values("seconds_remaining")

        active_view = active_view.rename(
            columns={
                "order_id": "ID",
                "dish_name": "Danie",
                "dish_category": "Kategoria",
                "quantity": "Liczba",
                "actual_preparation_seconds": "Czas [s]",
                "seconds_remaining": "Pozostało [s]",
                "is_delayed": "Opóźnione",
                "is_promotion": "Promocja"
            }
        )

        display(active_view)
    else:
        print("Brak aktywnych zamówień.")

    print("\n✅ OSTATNIE ZAKOŃCZONE ZAMÓWIENIA")
    if len(completed_orders) > 0:
        completed_view = completed_orders[
            [
                "order_id",
                "dish_name",
                "dish_category",
                "actual_preparation_seconds",
                "delay_seconds",
                "order_value",
                "actual_ready_at"
            ]
        ].sort_values("actual_ready_at", ascending=False).head(10)

        completed_view = completed_view.rename(
            columns={
                "order_id": "ID",
                "dish_name": "Danie",
                "dish_category": "Kategoria",
                "actual_preparation_seconds": "Czas [s]",
                "delay_seconds": "Opóźnienie [s]",
                "order_value": "Wartość [zł]",
                "actual_ready_at": "Zakończono o"
            }
        )

        display(completed_view)
    else:
        print("Brak zakończonych zamówień.")

    print("\n🍕 TOP DANIA")
    top_dishes = (
        df.groupby("dish_name")["quantity"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
    )

    display(top_dishes.to_frame("Liczba sztuk"))

In [6]:
orders = []
order_number = 1

simulation_time_seconds = 300   # 5 minut
refresh_seconds = 3             # odświeżenie co 3 sekundy
orders_per_refresh = 1          # ile nowych zamówień przy każdym odświeżeniu

start_time = time.time()

while time.time() - start_time < simulation_time_seconds:
    for _ in range(orders_per_refresh):
        new_order = generate_order(order_number)
        orders.append(new_order)
        order_number += 1

    show_dashboard(orders)

    time.sleep(refresh_seconds)

print("Symulacja zakończona.")

🍽️ SYMULACJA DASHBOARDU RESTAURACJI
Czas aktualny: 20:01:55
Status operacyjny: 🔴 Czerwony — Ryzyko przeciążenia kuchni.
Wszystkie zamówienia: 36
Aktywne zamówienia: 20
Zakończone zamówienia: 16
Średni czas przygotowania: 59 s
Udział opóźnień: 63.9%
Przychód: 1673.90 zł

🔥 AKTYWNE ZAMÓWIENIA


,ID,Danie,Kategoria,Liczba,Czas [s],Pozostało [s],Opóźnione,Promocja
18,ORD_000019,Burger klasyczny,Burger,1,56,3.0,True,True
11,ORD_000012,Pizza Margherita,Pizza,1,79,5.0,True,True
12,ORD_000013,Pizza Margherita,Pizza,1,76,5.0,True,True
13,ORD_000014,Pizza Margherita,Pizza,1,74,6.0,True,True
14,ORD_000015,Pizza Pepperoni,Pizza,3,72,7.0,True,True
24,ORD_000025,Makaron Carbonara,Makaron,2,43,9.0,False,True
16,ORD_000017,Pizza Capricciosa,Pizza,1,69,10.0,False,True
15,ORD_000016,Pizza Margherita,Pizza,1,75,13.0,True,True
22,ORD_000023,Makaron Carbonara,Makaron,2,58,17.0,True,True
31,ORD_000032,Burger klasyczny,Burger,1,33,20.0,False,True



✅ OSTATNIE ZAKOŃCZONE ZAMÓWIENIA


,ID,Danie,Kategoria,Czas [s],Opóźnienie [s],Wartość [zł],Zakończono o
10,ORD_000011,Pizza Margherita,Pizza,74,14,29.9,2026-06-01 20:01:52.620648
19,ORD_000020,Burger BBQ,Burger,40,0,36.9,2026-06-01 20:01:46.238841
20,ORD_000021,Burger klasyczny,Burger,35,0,31.9,2026-06-01 20:01:44.315911
25,ORD_000026,Sałatka Cezar,Sałatka,17,0,27.9,2026-06-01 20:01:41.699347
9,ORD_000010,Burger BBQ,Burger,59,14,73.8,2026-06-01 20:01:34.581894
17,ORD_000018,Burger klasyczny,Burger,33,0,31.9,2026-06-01 20:01:33.008768
21,ORD_000022,Zupa dnia,Zupa,20,0,37.8,2026-06-01 20:01:32.377897
6,ORD_000007,Pizza Margherita,Pizza,66,6,29.9,2026-06-01 20:01:32.355710
8,ORD_000009,Makaron Bolognese,Makaron,59,4,34.9,2026-06-01 20:01:31.509044
5,ORD_000006,Makaron Carbonara,Makaron,68,18,32.9,2026-06-01 20:01:31.315302



🍕 TOP DANIA


,Liczba sztuk
dish_name,
Makaron Carbonara,12
Pizza Margherita,8
Burger BBQ,7
Pizza Pepperoni,7
Burger klasyczny,6


KeyboardInterrupt: 